# Model Evaluation
Load saved models and visualize performance on the test set.

In [1]:
import joblib
import polars as pl
from sklearn.metrics import precision_score, recall_score
from engine import load_and_prepare_data
from visualizations import (
    plot_confusion_matrix,
    plot_cross_validation_scores,
    plot_optuna_trials,
    plot_metrics_comparison,
)

/Users/vincent/Labs/cancer-detection/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and models

In [2]:
_, features_test, _, target_test, _ = load_and_prepare_data()

decision_tree = joblib.load("models/decision_tree.joblib")
knn           = joblib.load("models/knn.joblib")

tree_predictions = decision_tree.predict(features_test)
knn_predictions  = knn.predict(features_test)

## Decision Tree

In [3]:
# Rerun cross-validation to get fold scores for plotting
from sklearn import model_selection, tree

cross_validation_scores = model_selection.cross_validate(
    decision_tree, features_test, target_test,
    cv=5, scoring=["accuracy"], return_train_score=True
)

plot_cross_validation_scores(cross_validation_scores, "Decision Tree").show()
plot_confusion_matrix(target_test, tree_predictions, "Decision Tree").show()

## KNN

In [4]:
# Reload the Optuna study if saved, or re-run tuning
# If you saved the study: study = joblib.load("models/knn_study.joblib")
# Otherwise re-tune (quick — 40 trials):
import optuna
from sklearn import model_selection, neighbors

optuna.logging.set_verbosity(optuna.logging.WARNING)

_, features_train, _, target_train, _ = load_and_prepare_data()

def objective(trial):
    k = trial.suggest_int("n_neighbors", 2, 41)
    model = neighbors.KNeighborsClassifier(n_neighbors=k)
    return model_selection.cross_val_score(model, features_train, target_train, cv=5, scoring="accuracy").mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)

plot_optuna_trials(study, "KNN").show()
plot_confusion_matrix(target_test, knn_predictions, "KNN").show()

## Model Comparison

In [5]:
comparison = pl.DataFrame({
    "Model":     ["Decision Tree", "KNN"],
    "Accuracy":  [precision_score(target_test, p, average="weighted") for p in [tree_predictions, knn_predictions]],
    "Precision": [precision_score(target_test, p, average="weighted") for p in [tree_predictions, knn_predictions]],
    "Recall":    [recall_score(target_test, p, average="weighted")    for p in [tree_predictions, knn_predictions]],
})

print(comparison)

plot_metrics_comparison(
    comparison["Model"].to_list(),
    comparison["Accuracy"].to_list(),
    comparison["Precision"].to_list(),
    comparison["Recall"].to_list(),
).show()

shape: (2, 4)
┌───────────────┬──────────┬───────────┬──────────┐
│ Model         ┆ Accuracy ┆ Precision ┆ Recall   │
│ ---           ┆ ---      ┆ ---       ┆ ---      │
│ str           ┆ f64      ┆ f64       ┆ f64      │
╞═══════════════╪══════════╪═══════════╪══════════╡
│ Decision Tree ┆ 0.947368 ┆ 0.947368  ┆ 0.947368 │
│ KNN           ┆ 0.947368 ┆ 0.947368  ┆ 0.947368 │
└───────────────┴──────────┴───────────┴──────────┘
